# H-001 · Pooling (sector vs industry)

Form pairs only inside the same Yahoo GICS **sector** or **industry** leaf.
Membership is PIT; labels are current Yahoo GICS. Soft freeze book N=35, default windows,
`break_mode=off`. Freeze `POOL_STAR` on research IS walk-forward only.

Do not open the sealed holdout in this notebook.

Next: run H-002.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    apply_bh_fdr,
    candidates_path,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    pairs_path,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.book import select_book

VERSION = "v2"
DATA_DIR = s2_data_dir(ROOT, version=VERSION)
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
os.makedirs(os.path.dirname(STAR_PATH), exist_ok=True)
cuts = load_s2_cuts(root=ROOT, version=VERSION)
DROP_RATE = cuts.get("yahoo_drop_rate")
print("STAR_PATH", STAR_PATH)
print("yahoo_drop_rate", DROP_RATE)
print("RESEARCH_IS_END", cuts.get("RESEARCH_IS_END"))

stack = load_star_stack(STAR_PATH) if os.path.isfile(STAR_PATH) else {}
stack["RESEARCH_IS_END_STAR"] = cuts["RESEARCH_IS_END"]
SOFT_N = 35
BAR = "1d"
POOL_ARMS = ("sector", "industry")
HYP_ID = "H-001"
print("soft N", SOFT_N, "arms", POOL_ARMS)


## 1. Load candidates + build soft books


In [ ]:
cand = pd.read_csv(candidates_path(BAR, version=VERSION, root=ROOT))
cand["pvalue"] = pd.to_numeric(cand["pvalue"], errors="coerce")

def book_for(pool: str, n: int = SOFT_N) -> list[str]:
    if pool == "industry":
        mask = cand["same_industry"].astype(bool)
    else:
        mask = cand["same_sector"].astype(bool)
    sub = cand.loc[mask & cand["eligible"].astype(bool) & cand["pvalue"].notna()].copy()
    # pool key for select_book caps: industry or sector leaf of y
    leaf_col = "industry_y" if pool == "industry" else "sector_y"
    pool_map = {str(r.pair_id): str(getattr(r, leaf_col)) for r in sub.itertuples(index=False)}
    # Large per-pool cap so global N binds for S&P leaves
    chosen = select_book(
        sub,
        pool_of_pair=pool_map,
        per_pool_cap=max(n, 10),
        global_cap=n,
        pvalue_threshold=0.05,
    )
    return [str(p) for p in chosen["pair_id"]]

books = {arm: book_for(arm) for arm in POOL_ARMS}
for arm, ids in books.items():
    print(arm, "n=", len(ids), "head=", ids[:5])

# STAR book (PAIRS_STAR) is written in the freeze cell after typing POOL_STAR.
for arm, ids in books.items():
    print(arm, "ready", len(ids))


## 2. Load panels and score arms


In [ ]:
# Union of pairs across arms so one panel load covers both.
union_ids = sorted(set(books["sector"]) | set(books["industry"]))
train, full = load_universe_panels(None, BAR, union_ids, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(BAR)
train = overlay_ols_hedge(train, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                          hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
full = overlay_ols_hedge(full, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                         hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
stack["BAR_STAR"] = BAR
is_end = pd.Timestamp(cuts["RESEARCH_IS_END"])
is_panel = train.copy()
oos_panel = full.loc[pd.to_datetime(full["date"]) > is_end].copy()
s1_weekly = load_s1_weekly(ROOT)
print("IS rows", len(is_panel), "OOS rows", len(oos_panel), "is_end", is_end)

def filter_panel(panel, pair_ids):
    return panel.loc[panel["pair_id"].astype(str).isin(set(pair_ids))].copy()

configs = {}
for arm in POOL_ARMS:
    cfg = config_from_stack(stack, bar=BAR, break_mode="off")
    configs[arm] = cfg

# Score each arm on its own book by restricting panel rows inside fold_val via separate runs.
# fold_val_metrics expects one panel; run per arm and concat.
register_hypothesis_arms(HYP_ID, list(POOL_ARMS), overwrite=True, path=os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json"))
fold_parts = []
full_parts = []
dates = pd.DatetimeIndex(pd.to_datetime(is_panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
display(fold_table(folds))
for arm in POOL_ARMS:
    p = filter_panel(is_panel, books[arm])
    fd = fold_val_metrics(p, folds, {arm: configs[arm]}, hyp_id=HYP_ID, s1_weekly=s1_weekly)
    fi = full_is_metrics(p, {arm: configs[arm]}, hyp_id=HYP_ID, s1_weekly=s1_weekly)
    fold_parts.append(fd)
    full_parts.append(fi)
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-001 pooling fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
display(arm_selection_table(fold_df, full_is_df))


## 3. Type `POOL_STAR`


In [ ]:
POOL_STAR = ""  # type after the table: "sector" | "industry"
require_star("POOL_STAR", POOL_STAR)
stack = load_star_stack(STAR_PATH) if os.path.isfile(STAR_PATH) else {}
stack["POOL_STAR"] = POOL_STAR
stack["RESEARCH_IS_END_STAR"] = cuts["RESEARCH_IS_END"]
stack["BAR_STAR"] = stack.get("BAR_STAR") or "1d"
stack["PAIRS_STAR"] = books[POOL_STAR]
stack["N_STAR"] = stack.get("N_STAR") or SOFT_N
# Write frozen pairs CSV for later hyps.
pd.DataFrame({"pair_id": books[POOL_STAR], "rank": range(1, len(books[POOL_STAR]) + 1)}).to_csv(
    pairs_path(BAR, version=VERSION, root=ROOT), index=False
)
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "POOL_STAR", POOL_STAR, "n_pairs", len(books[POOL_STAR]))


## 4. Notes for next hyp

Next: run H-002.
